In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

**Load Data**

In [ ]:
pull_data = "https://raw.githubusercontent.com/cdavidshaffer/CPSC4970-AI/master/data/m3train.csv"

data_import = pd.read_csv(pull_data)

# Drop rows with missing values
data_import = data_import.dropna()

# Take a look at data
print(data_import.shape)
data_import.head()

**Naive grid search implementation**

In [ ]:
train, test = train_test_split(
    data_import,
    test_size=0.33,
    random_state=0
)

#train on all except last column
X_train = train.iloc[:, :-1]
X_test = test.iloc[:, :-1]
y_train = train.iloc[:, -1]
y_test = test.iloc[:, -1]

# all rows / columns except target column
X = data_import.iloc[:, :-1]

# only the last column
y = data_import.iloc[:, -1]

**Add pipeline**

In [ ]:
pl = Pipeline([
    ("poly", PolynomialFeatures()),
    ("norm", StandardScaler()),
    ("model", TransformedTargetRegressor(
        regressor=Lasso(alpha=0.1, max_iter=100000),
        transformer=StandardScaler()
    ))
])

# train 

pl.fit(X_train, y_train)

p_train = pl.predict(X_train)
p_test = pl.predict(X_test)

**Print training values**

In [ ]:
print("Training Data Summary:")
print(pd.DataFrame({
    "Actual": y_train.values[:10],
    "Predicted": p_train[:10]
}))

**Print testing values**

In [ ]:
print("Testing Data Summary:")
print(pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": p_test[:10]
}))

**Print Metrics to Compare against Cross Val / GridSearch**

In [ ]:
mse_train = mean_squared_error(y_train, p_train)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, p_train)

mse_test = mean_squared_error(y_test, p_test)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, p_test)

print("Training MSE:", mse_train)
print("Training RMSE:", rmse_train)
print("Training r^2:", r2_train)

print("Testing MSE:", mse_test)
print("Testing RMSE:", rmse_test)
print("Testing r^2:", r2_test)

**Cross validation section**

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(pl, X, y, cv=kfold,
                        scoring="neg_root_mean_squared_error")

# convert scores back to positive values
rmse_scores = -scores


**RMSE Before GridSearch is Implemented**

In [ ]:
print("RMSE scores: ", rmse_scores)
print("Avg RMSE: ", rmse_scores.mean())

**Grid Search w/ Poly_degree, Lasso, & Ridge Parameter Options**

In [ ]:
# set parameters
#giving options for lasso & ridge

param_grid = [
    {
        "poly__degree": [1, 2, 3],
        "model__regressor": [Lasso(max_iter=100000)],
        "model__regressor__alpha": [0.01, 0.1, 1, 10, 100]
    },
    {
        "poly__degree": [1, 2, 3],
        "model__regressor": [Ridge()],
        "model__regressor__alpha": [0.01, 0.1, 1, 10, 100]
    }
]

# grid search for best ones
grid = GridSearchCV(
    pl,
    param_grid,
    cv=kfold,
    scoring="neg_root_mean_squared_error"
)


grid.fit(X, y)

#set model variable for grader
model = grid.best_estimator_

# pull best model type 
best_regressor = grid.best_params_["model__regressor"]


**Optimal Hyperparameters found by GridSearchCV**

In [ ]:
print("Best regularization type:", type(best_regressor).__name__)
print("Best degree:", grid.best_params_["poly__degree"])
print("Best alpha:", grid.best_params_["model__regressor__alpha"])
print("Best RMSE:", -grid.best_score_)

Based on the above, all predictions are plus or minus 23 (rounded up from 22.8 above)

**Grading Section**

In [ ]:
X_grading = pd.read_csv('https://raw.githubusercontent.com/cdavidshaffer/CPSC4970-AI/master/data/m3test.csv')
predicted = model.predict(X_grading)

print(predicted)
print(len(predicted))

**Final Thoughts**

Tuning the parameters is hard when you're not that familiar with the data or have a deep understanding of how the parameters work. The last module seemed easier as the housing data was more intuitive to work with and you could evaluate it with more common sense and reasoning then a from a purely mathematical standpoint like abstract data above.

The good news is you can check the output easily with the RMSE, and it's simple enough to keep playing with the parameters to see what makes the error estimate go down. I originally started with a poly degree option of 1-6, and just Lasso. The best RMSE I achieved was through adding Ridge, which GridSearch chose along with a poly degree of 2 and alpha of 1.